In [70]:
from datasets import load_dataset

In [71]:
import torch
from torch import nn
from peft import LoraConfig, get_peft_model, PeftModel

In [72]:
net = nn.Sequential(
    nn.Linear(10, 10),
    nn.ReLU(),
    nn.Linear(10, 5)
)
print(net)

Sequential(
  (0): Linear(in_features=10, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=5, bias=True)
)


In [73]:
# 针对第一层模型的参数调整
config = LoraConfig(target_modules=["0"])
print(config)

LoraConfig(task_type=None, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'0'}, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [74]:
peft_model = get_peft_model(net, config)
peft_model

PeftModel(
  (base_model): LoraModel(
    (model): Sequential(
      (0): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=10, bias=True)
        (lora_dropout): ModuleDict(
          (default): Identity()
        )
        (lora_A): ModuleDict(
          (default): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (default): Linear(in_features=8, out_features=10, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
        (lora_magnitude_vector): ModuleDict()
      )
      (1): ReLU()
      (2): Linear(in_features=10, out_features=5, bias=True)
    )
  )
)

In [75]:
# 保存调整了第一层模型的模型
peft_model.save_pretrained("./customer_lora_1") 

In [76]:
# 调整第二层参数
config2 = LoraConfig(target_modules=["2"])
print(config2)


LoraConfig(task_type=None, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'2'}, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [77]:
peft_model2 = get_peft_model(net, config2)
print(peft_model2)


PeftModel(
  (base_model): LoraModel(
    (model): Sequential(
      (0): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=10, bias=True)
        (lora_dropout): ModuleDict(
          (default): Identity()
        )
        (lora_A): ModuleDict(
          (default): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (default): Linear(in_features=8, out_features=10, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
        (lora_magnitude_vector): ModuleDict()
      )
      (1): ReLU()
      (2): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=5, bias=True)
        (lora_dropout): ModuleDict(
          (default): Identity()
        )
        (lora_A): ModuleDict(
          (default): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (default): Linear(in_features=8, out_features=5, bia

In [78]:
peft_model2.save_pretrained("./customer_lora_2")

## 加载已训练好的模型

In [79]:
# 重新定义一个网络层
net_new = nn.Sequential(
    nn.Linear(10, 10),
    nn.ReLU(),
    nn.Linear(10, 5)
)
print(net_new)

Sequential(
  (0): Linear(in_features=10, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=5, bias=True)
)


In [80]:
# 加载已训练好的模型
model_loaded = PeftModel.from_pretrained(net_new,  model_id="./customer_lora_1", adapter_name="CLoraA")
print(model_loaded)

PeftModel(
  (base_model): LoraModel(
    (model): Sequential(
      (0): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=10, bias=True)
        (lora_dropout): ModuleDict(
          (CLoraA): Identity()
        )
        (lora_A): ModuleDict(
          (CLoraA): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (CLoraA): Linear(in_features=8, out_features=10, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
        (lora_magnitude_vector): ModuleDict()
      )
      (1): ReLU()
      (2): Linear(in_features=10, out_features=5, bias=True)
    )
  )
)


In [81]:
model_loaded.load_adapter(model_id="./customer_lora_2", adapter_name="CLoraB")
print(model_loaded)

PeftModel(
  (base_model): LoraModel(
    (model): Sequential(
      (0): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=10, bias=True)
        (lora_dropout): ModuleDict(
          (CLoraA): Identity()
        )
        (lora_A): ModuleDict(
          (CLoraA): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (CLoraA): Linear(in_features=8, out_features=10, bias=False)
        )
        (lora_embedding_A): ParameterDict()
        (lora_embedding_B): ParameterDict()
        (lora_magnitude_vector): ModuleDict()
      )
      (1): ReLU()
      (2): lora.Linear(
        (base_layer): Linear(in_features=10, out_features=5, bias=True)
        (lora_dropout): ModuleDict(
          (CLoraB): Identity()
        )
        (lora_A): ModuleDict(
          (CLoraB): Linear(in_features=10, out_features=8, bias=False)
        )
        (lora_B): ModuleDict(
          (CLoraB): Linear(in_features=8, out_features=5, bias=Fals

In [82]:
model_loaded.active_adapter

'CLoraA'

In [83]:
model_loaded(torch.arange(0,10).view(1,10).float())


tensor([[ 0.2850, -0.1459,  0.1780,  0.3604, -1.6640]])

In [84]:
# 查看当前加载的模型中有哪些参数
for name, param in model_loaded.named_parameters():
    print(name, param)

base_model.model.0.base_layer.weight Parameter containing:
tensor([[ 4.0756e-02, -1.2393e-01, -2.4489e-01, -2.2566e-01,  1.8619e-01,
          1.5270e-01,  2.6841e-01,  2.0076e-01,  1.6929e-01,  2.7050e-01],
        [-1.1701e-01, -7.4529e-02,  2.5321e-01,  2.2453e-01,  3.0356e-01,
          2.3940e-01, -2.3762e-01, -1.8469e-01, -2.7066e-01,  1.8309e-01],
        [ 1.9076e-01,  1.0581e-01,  2.3965e-01, -2.4399e-01, -3.0464e-03,
          1.3916e-01,  6.5902e-03,  1.0577e-01, -2.7412e-02, -6.1162e-02],
        [-8.2693e-03,  1.7451e-02,  2.7769e-01,  3.0737e-01, -3.0125e-01,
         -2.7912e-01, -1.0632e-01, -9.9988e-02, -9.8244e-02,  1.8968e-01],
        [-4.2009e-02, -4.6170e-02, -2.0759e-01, -2.5872e-01,  2.3387e-01,
         -1.6499e-01,  2.3546e-01, -1.2232e-01, -2.6806e-01,  3.1290e-01],
        [ 1.9180e-01, -1.1516e-01, -7.9140e-02, -1.5712e-01,  2.3667e-01,
         -2.3817e-01, -3.0185e-01, -2.9081e-02, -3.0157e-02, -1.1258e-01],
        [-2.7191e-01, -3.0275e-01, -2.4475e-01,

In [85]:
# 查看当前加载的模型中有哪些参数
for name, param in model_loaded.named_parameters():
    if name in ['base_model.model.0.lora_A.CLoRA1.weight','base_model.model.0.lora_B.CLoRA1.weight']:
        param.data = torch.ones_like(param)

In [86]:
# 在进行正向传播
model_loaded(torch.arange(0,10).view(1,10).float())

tensor([[ 0.2850, -0.1459,  0.1780,  0.3604, -1.6640]])

In [88]:
# 切换适配器
model_loaded.set_adapter("CLoraB")
model_loaded(torch.arange(0, 10).view(1, 10).float())

tensor([[ 0.2850, -0.1459,  0.1780,  0.3604, -1.6640]], grad_fn=<AddBackward0>)